# Ejercicio 1: registrar cada corrida con MLflow

`viajes_features.csv`, el de la Semana 10, con la preparación importada de `preparacion_taxis.py`. La sesión pasada, cada modelo quedaba en una fila de una tabla en memoria: al reiniciar el kernel, la tabla desaparecía, y nada decía con qué datos, qué división ni qué hiperparámetros salió cada número. Hoy cada corrida queda registrada en el servidor de MLflow del Codespace, que guarda el registro aunque reinicies el kernel o cierres el notebook.

Corre todas las celdas antes de tocar nada (**Run All**): corre de principio a fin sin los `TODO` resueltos. Cada vez que corres una celda que entrena, MLflow registra una corrida nueva, también si es una repetición: un registro de experimentos no borra nada. Para empezar de cero, abre la interfaz de MLflow (pestaña **Ports**, puerto 5050), entra al experimento, selecciona todas sus corridas y bórralas con **Delete**.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from preparacion_taxis import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, dividir, preparacion

# MLflow avisa en cada corrida que el modelo se guarda sin firma (la
# descripcion de sus columnas de entrada). La firma importa al servir el
# modelo, en la Semana 12; hoy solo ensucia la salida.
logging.getLogger("mlflow").setLevel(logging.ERROR)

viajes = pd.read_csv("viajes_features.csv")
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (2380, 10) Prueba: (595, 10)


## La huella de los datos

In [2]:
def huella(ruta):
    """Los primeros 12 caracteres del hash MD5 del archivo. Si cambia un
    solo byte del CSV, la huella cambia."""
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella("viajes_features.csv")
print("Huella de viajes_features.csv:", huella_datos)

Huella de viajes_features.csv: 4800204d50c0


## Paso 1: el experimento

In [3]:
# El registro vive en el servidor de MLflow del Codespace (puerto 5050). La
# direccion viene de la variable de entorno MLFLOW_TRACKING_URI, como en un
# equipo de datos: el codigo no la escribe. Sin esa variable (en tu laptop),
# MLflow guarda el registro en una carpeta mlruns al lado del notebook.
print("Registro de MLflow:", mlflow.get_tracking_uri())

experimento= mlflow.set_experiment("tarifa-taxis")
print("Experimento:", experimento.name)

Registro de MLflow: http://127.0.0.1:5050
Experimento: tarifa-taxis


## Paso 2: entrenar y registrar

In [4]:
def evaluar(nombre, modelo, X_ent=X_ent, X_prueba=X_prueba, y_ent=y_ent, y_prueba=y_prueba, semilla_division=42):
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({**modelo.get_params(), "semilla_division": semilla_division,
                           "huella_datos": huella_datos, "filas_entrenamiento": len(X_ent)})
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba, "segundos": segundos})
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre}: entrenamiento {mae_ent:.6f}; prueba {mae_prueba:.6f}")


## Paso 3: los dos modelos de la Semana 10

In [5]:
evaluar("bosque aleatorio", RandomForestRegressor(n_estimators=200, random_state=42), X_ent, X_prueba, y_ent, y_prueba)
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42), X_ent, X_prueba, y_ent, y_prueba)

bosque aleatorio: entrenamiento 0.473949; prueba 1.301589


gradient boosting: entrenamiento 1.070539; prueba 1.275352


## Paso 4: la corrida de tu compañero

In [6]:
# Un companero entrena "el mismo" gradient boosting, sobre el mismo CSV,
# pero hace su propia division con otra semilla.
X_ent_c, X_prueba_c, y_ent_c, y_prueba_c = train_test_split(
    viajes[NUMERICAS + YA_LISTAS + CATEGORICAS], viajes[OBJETIVO], test_size=0.2, random_state=7
)
evaluar("gradient boosting", GradientBoostingRegressor(random_state=42), X_ent_c, X_prueba_c, y_ent_c, y_prueba_c, semilla_division=7)

gradient boosting: entrenamiento 1.079282; prueba 1.186727


## Paso 5: lo que quedó registrado

In [7]:
experimento = mlflow.get_experiment_by_name("tarifa-taxis")
if experimento is None:
    print("Todavia no existe el experimento tarifa-taxis.")
else:
    corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
    columnas = ["run_id", "tags.mlflow.runName", "params.semilla_division", "params.huella_datos", "params.n_estimators", "metrics.mae_prueba", "metrics.segundos"]
    columnas = [columna for columna in columnas if columna in corridas.columns]
    if len(corridas) == 0:
        print("El experimento tarifa-taxis no tiene corridas.")
    else:
        print(corridas[columnas].to_string(index=False))

                          run_id tags.mlflow.runName params.semilla_division params.huella_datos params.n_estimators  metrics.mae_prueba  metrics.segundos
8c54c8d3f5384c26acc6b6fce26572ea   gradient boosting                       7        4800204d50c0                 100            1.186727          0.116568
fdb2f6a14fd946bfb9c512f694d73259   gradient boosting                       7        4800204d50c0                 100            1.186727               NaN
8f85f9f8fd2546189972cff8263cedb2   gradient boosting                      42        4800204d50c0                 100            1.275352          0.121416
70830bfb03fc48489ec3e7b6d17f50e2   gradient boosting                      42        4800204d50c0                 100            1.275352               NaN
a2126b2dd76f422bb3ba756c143c5ff3    bosque aleatorio                      42        4800204d50c0                 200            1.301589          0.849634
bad7297ae2fa46fdaad462f3de4c9a80    bosque aleatorio                  